In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from pathlib import Path

plots = Path("plots")

In [4]:
from matplotlib import pyplot as plt

def plot_ensemble_mean_spread(mean: xr.Dataset, spread: xr.Dataset, plot: Path, sample: slice = slice(None)):
    fig, ax = plt.subplots()

    months = sorted(set(zip(
        pd.to_datetime(mean.valid_time).year,
        pd.to_datetime(mean.valid_time).month,
    )))
    colours = ["tab:orange", "tab:blue"]
    
    ax.set_title(mean.long_name)
    ax.set_xlabel(f"mean [{mean.units}]")
    ax.set_ylabel(f"spread [{spread.units}]")

    for (year, month), colour in zip(months, colours):
        ax.scatter(
            mean.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            spread.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            s=1, label=f"{year}/{month:02}", edgecolor="none", marker=".", c=colour,
        )

    fig.legend(loc="upper right", markerscale=10)

    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)

    plt.close(fig)

In [5]:
from tqdm import tqdm

In [ ]:
for key in tqdm(ds_single.mean):
    plot_ensemble_mean_spread(ds_single.mean[key], ds_single.spread[key], plots / "single" / f"{key}.png", slice(0, None, 1000))

  3%|▎         | 7/226 [00:30<16:53,  4.63s/it]

In [ ]:
for key in tqdm(ds_pressure.mean):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        plot_ensemble_mean_spread(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            ds_pressure.spread[key].sel(pressure_level=pressure),
            plots / "pressure" / f"{key}-{pressure}hPa.png",
            slice(0, None, 1000),
        )